# Evaluate the model

Use the JuniperMapper fieldwork data to validate the trained model. To evaluate the overall model performance, three metrics were used:

1. Precision metric: assess how precise the model is in delineating the shrubs
2. Recall: evaluate how accurate the model is in recalling the shrubs
3. F1-score: examine the ability of the model to maintain the trade-off precision–recall.

$$
Precision = \frac{TP}{TP+FP}
$$

$$
Recall = \frac{TP}{TP+FN}
$$

$$
F1 = \frac{2PR}{P+R}
$$

For Mask R-CNN, there are two useful interpretations of Precision, Recall, and F1:

1. Pixel-level segmentation metrics: measure how precisely the predicted masks delineate shrub pixels.
2. Instance-level detection metrics: measure whether individual shrubs are found, using a mask-IoU threshold.

This code reporting both, although the reference paper ([Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191)) report only instance-level detection.

In [1]:
import os
import sys

import matplotlib.pyplot as plt

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'bin'))
import mrcnn.model as modellib
from shrubs import ShrubDataset
from shrubs import ShrubInferenceConfig
from shrubs import evaluate_pixel_metrics
from shrubs import predict_dataset
from shrubs import compute_iou_range
from shrubs import compute_siou_range
from shrubs import plot_performance

Using TensorFlow backend.


## Configurations

In [4]:
%matplotlib inline 

ROOT = os.path.join(os.path.dirname(os.getcwd()))

# Directory to save logs and trained model
MODEL_DIR = os.path.join(ROOT, "data/logs")

MODEL_VERSION = 1

# Directory with test data
VAL_DIR_PI = os.path.join(
    ROOT,
    "data/JuniperMapper", "Photo_Interpretation_Data/Test"
)

VAL_DIR_FW = os.path.join(
    ROOT,
    "data/JuniperMapper", "Field_Work_Data/External_Val_Data"
)

# Directory to save evaluation stats
STATS_DIR = os.path.join(ROOT, "results/evaluation")

In [3]:
# Validation datasets
dataset_val_pi = ShrubDataset()
dataset_val_pi.load_dataset(
    os.path.join(VAL_DIR_PI, "Images"),
    os.path.join(VAL_DIR_PI, "masks_points.json")
)
dataset_val_pi.prepare()

dataset_val_fw = ShrubDataset()
dataset_val_fw.load_dataset(
    os.path.join(VAL_DIR_FW, "Images"),
    os.path.join(VAL_DIR_FW, "masks_points.json")
)
dataset_val_fw.prepare()

In [5]:
config = ShrubInferenceConfig()

# Create model in testing mode
model = modellib.MaskRCNN(
    mode="inference",
    config=config,
    model_dir=MODEL_DIR
)

# Get path to saved weights
# Either set a specific path or find last trained weights
model_path = os.path.join(MODEL_DIR, f"mask_rcnn_juniper_v{MODEL_VERSION}.h5")
# Or find the last trained weights
# model_path = model.find_last()

# Load trained weights
print("Loading weights from ", model_path)
model.load_weights(model_path, by_name=True)

Loading weights from  c:\IRANZO\shrubs-segmentation\data/logs\mask_rcnn_juniper_v1.h5


## Pixel-level metrics

Combine all shrub-instance masks in an image into one binary shrub mask. Then count the three metrics to assess overall model performance. `TP`, `FP`, and `FN` are numbers of pixels.

In [ ]:
pixel_results, pixel_results_per_image = evaluate_pixel_metrics(
    model=model,
    dataset_test=dataset_val_pi,
    shrub_class_id=1,
    confidence_threshold=0.5
)

print("Pixel-level evaluation for PI validation set")
print(f"Precision: {pixel_results['precision']:.4f}")
print(f"Recall:    {pixel_results['recall']:.4f}")
print(f"F1-score:  {pixel_results['f1']:.4f}")
print(f"TP pixels: {pixel_results['tp_pixels']}")
print(f"FP pixels: {pixel_results['fp_pixels']}")
print(f"FN pixels: {pixel_results['fn_pixels']}")


These metrics answer:

* **Pixel Precision:** Of all pixels labelled as shrubs, how many really belong to shrubs?
* **Pixel Recall:** Of all annotated shrub pixels, how many were recovered?
* **Pixel F1:** How well does the model balance over-segmentation and under-segmentation?

## Instance-level metrics

For instance-level evaluation, a predicted shrub is a true positive when:

* its class is shrub
* it has not already been matched to another ground-truth shrub
* its mask IoU with a ground-truth shrub is at least the selected threshold

The function `mrcnn.utils.compute_matches()` calculates mask overlaps and performs one-to-one matching between predictions and ground-truth instances. It uses mask IoU rather than bounding-box IoU, so this evaluation considers the quality of the predicted segmentation when deciding whether a shrub was detected correctly.

Note: Mask R-CNN repository returns instance masks in `r["masks"]`, class IDs in `r["class_ids"]`, confidence scores in `r["scores"]`, and boxes in `r["rois"]`.

### Iterate between several confident thresholds

Use a declared pair of thresholds.

* The **confidence threshold** controls which predicted instances are included.
* The **mask-IoU threshold** controls how closely a predicted mask must overlap an annotation to count as a correct shrub instance. Higher IoU thresholds impose stricter delineation requirements.

Evaluate reports results at several IoU thresholds and save them to a csv file.

In [30]:
dataset_name = "khaldi_val_fw"
dataset_obj = dataset_val_fw
ev_metric_threshold = 0.5

To avoid repeating inference across both evaluations, use one shared cache.

Run the following chunk only when the `dataset_obj` is changed.

In [31]:
predictions = predict_dataset(
    model,
    dataset_obj,
    verbose=False,
)

In [24]:
out_csv_iou_file = os.path.join(
    STATS_DIR,
    f"model_config_v{MODEL_VERSION}",
    dataset_name,
    f"iou_threshold_{int(ev_metric_threshold*100)}.csv"
)


In [25]:
rows = compute_iou_range(
    model=None,
    dataset=dataset_obj,
    out_csv_file=out_csv_iou_file,
    iou_threshold=ev_metric_threshold,
    predictions=predictions,
)

IoU=0.75, confidence=0.00, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.05, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.10, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.15, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.20, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.25, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.30, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.35, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.40, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.45, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.50, predictions=992, P=0.4103, R=0.5899, F1=0.4839
IoU=0.75, confidence=0.55, predictions=946, P=0.4292, R=0.5884, F1=0.4963
IoU=0.75, confidence=0.60, predictions=902, P=0.4479, R=0.5855, F1=0.5075
IoU=0.75, confidence=0.65, predictions

The paper where JuniperMapper dataset is evaluated ([Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191)) used also a S-IoU metric to assess the instance-level metrics. The S-IoU:

> evaluates the proportion of shrubs' areas being detected by the model since they divide the intersection by the ground truth area and they use all matching ground truth shrubs instead of the best matching shrub. This metric evaluates the predictions and labels differently. The first equation was used to identify the number of TPs and FPs, while the second was used to identify the number of FNs.

$$
\mathrm{S\text{-}IoU}(p, S_{l_{\mathrm{match}}})
=
\frac{
\operatorname{Area}\left(
p \cap \bigcup_{k=1}^{\left|S_{l_{\mathrm{match}}}\right|} l_k
\right)
}{
\operatorname{Area}\left(
\bigcup_{k=1}^{\left|S_{l_{\mathrm{match}}}\right|} l_k
\right)
}
$$

$$
\mathrm{S\text{-}IoU}(l, S_{p_{\mathrm{match}}})
=
\frac{
\operatorname{Area}\left(
l \cap \bigcup_{k=1}^{\left|S_{p_{\mathrm{match}}}\right|} p_k
\right)
}{
\operatorname{Area}(l)
}
$$

where: $S_{p_{\mathrm{match}}}$ the set of all matching predictions. $S_{l_{\mathrm{match}}}$ is the set of all matching labels. $l_k$ and $p_k$ are the label and prediction of id $k$, respectively.

We defined a new function to address this new metric, computing the two independent calculations required by S-IoU metric:

- $\mathrm{S\text{-}IoU}(p, S_{l_{\mathrm{match}}})$: determines whether each prediction is TP or FP.
- $\mathrm{S\text{-}IoU}(l, S_{p_{\mathrm{match}}})$: determines whether each ground-truth instance is detected or is an FN.

The custom function `compute_s_iou_matches` is able to match one label to several predictions and several labels to one prediction. The new implementation defines matching instances as same-class masks with positive overlap.

> Note: The [Khaldi et al. 2024](https://doi.org/10.1016/j.jag.2024.104191) paper uses the prediction-directed TP count in both precision and recall, even though FN is obtained independently through label-directed evaluation.

Paper-compatible interpretation:

| | |
|------------|----------------|
|Matching set | All same-class masks with at least one intersecting pixel.|
|S-IoU threshold|Applied after unioning matching masks.|
|Prediction evaluation|S-IoU(p, matching labels) >= threshold -> TP; Otherwise -> FP |
|Label evaluation|S-IoU(l, matching predictions) < threshold -> FN|
|Precision | TP / (TP + FP)|
|Recall | TP / (TP + FN) |

In [32]:
out_csv_siou_file = os.path.join(
    STATS_DIR,
    f"model_config_v{MODEL_VERSION}",
    dataset_name,
    f"siou_threshold_{int(ev_metric_threshold * 100)}.csv"
)

In [33]:
rows = compute_siou_range(
    model=None,
    dataset=dataset_obj,
    out_csv_file=out_csv_siou_file,
    siou_threshold=ev_metric_threshold,
    predictions=predictions,
)

S-IoU=0.50, confidence=0.00, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.05, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.10, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.15, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.20, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.25, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.30, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.35, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.40, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.45, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.50, predictions=1600, P=0.6150, R=0.6138, F1=0.6144
S-IoU=0.50, confidence=0.55, predictions=1513, P=0.6371, R=0.5985, F1=0.6172
S-IoU=0.50, confidence=0.60, predictions=1429, P=0.6585, R=0.5861, F1=0.6202

### Plot the results

In [34]:
csv_files = {
    0.50: {
        "IoU": os.path.join(
            STATS_DIR,
            f"model_config_v{MODEL_VERSION}",
            dataset_name,
            f"iou_threshold_50.csv"
        ),
        "S-IoU":os.path.join(
            STATS_DIR,
            f"model_config_v{MODEL_VERSION}",
            dataset_name,
            f"siou_threshold_50.csv"
        )
    },
    0.75: {
        "IoU": os.path.join(
            STATS_DIR,
            f"model_config_v{MODEL_VERSION}",
            dataset_name,
            f"iou_threshold_75.csv"
        ),
        "S-IoU":os.path.join(
            STATS_DIR,
            f"model_config_v{MODEL_VERSION}",
            dataset_name,
            f"siou_threshold_75.csv"
        )
    }
}

# Store the graphics inside a custom folder
plots_path = os.path.join(STATS_DIR, f"model_config_v{MODEL_VERSION}", dataset_name)
plot_performance(csv_files, plots_path)

Figure saved to: c:\IRANZO\shrubs-segmentation\results\evaluation\model_config_v1\khaldi_val_fw\evaluation_threshold_50.png
Figure saved to: c:\IRANZO\shrubs-segmentation\results\evaluation\model_config_v1\khaldi_val_fw\evaluation_threshold_75.png


[WindowsPath('c:/IRANZO/shrubs-segmentation/results/evaluation/model_config_v1/khaldi_val_fw/evaluation_threshold_50.png'),
 WindowsPath('c:/IRANZO/shrubs-segmentation/results/evaluation/model_config_v1/khaldi_val_fw/evaluation_threshold_75.png')]